# Anomaly detection on healthy synthetic test runs

This parallels TEP notebook 2.1. It evaluates false alarms from scores of the endogenous process variables on held-out healthy runs using the models and alarm settings fitted before testing. The threshold is selected from validation data in notebook 1.1; this notebook does not adjust it using the test set.

In [1]:
from pathlib import Path
import sys


def _find_carl_project_root(start=Path.cwd()):
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the project root from the notebook working directory.")


PROJECT_ROOT = _find_carl_project_root()
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

Project root: <repo>


In [2]:
import numpy as np
import pandas as pd

from carl.causalai_process import ENDOGENOUS_NAMES


data_dir = PROJECT_ROOT / "data" / "interim" / "causalAIProcess"
source_dir = PROJECT_ROOT / "data" / "processed" / "causalAIProcess"
models_dir = PROJECT_ROOT / "models" / "causalAIProcess" / "max_lag_selection"
results_dir = PROJECT_ROOT / "reports" / "results" / "causalAIProcess"
results_dir.mkdir(parents=True, exist_ok=True)

test_file_name = "traces_scaled.parquet"
alarm_results_path = results_dir / "window_and_consecutive_alarm_selection.csv"
thresholds_path = results_dir / "alarm_thresholds_by_window.csv"
selected_lag_path = models_dir / "selected_max_lag.csv"
required_paths = [
    data_dir / test_file_name,
    source_dir / "runs.csv",
    source_dir / "causal_graph.csv",
    alarm_results_path,
    thresholds_path,
    selected_lag_path,
    models_dir / "sample_precision_train.npy",
]
missing_paths = [path for path in required_paths if not path.is_file()]
if missing_paths:
    raise FileNotFoundError(
        "Run notebooks 1.0 and 1.1 first. Missing: "
        + ", ".join(str(path) for path in missing_paths)
    )

df = pd.read_parquet(data_dir / test_file_name)
runs = pd.read_csv(source_dir / "runs.csv")
causal_graph = pd.read_csv(source_dir / "causal_graph.csv")
max_lag = int(pd.read_csv(selected_lag_path).loc[0, "best_max_lag"])
alarm_results = pd.read_csv(alarm_results_path)
thresholds_by_window = pd.read_csv(thresholds_path).set_index("Window Size")
f1_col = "Trajectory-Phase F1"
delay_col = "Median Anomaly Detection Delay"
best_f1 = alarm_results[f1_col].max()
best_candidates = alarm_results.loc[alarm_results[f1_col].eq(best_f1)].copy()
best_candidates = best_candidates.dropna(subset=[delay_col])
if best_candidates.empty:
    raise ValueError("No best-F1 configuration has a valid detection delay.")
best_idx = best_candidates[delay_col].idxmin()
best_settings = alarm_results.loc[best_idx]
window = int(best_settings["Window Size"])
consecutive_alarms = int(best_settings["Consecutive Alarms"])
threshold = float(thresholds_by_window.loc[window, "Threshold"])
if not np.isclose(threshold, best_settings["Threshold"]):
    raise ValueError("Selected validation threshold does not match the saved threshold table.")
sample_precision = np.load(models_dir / "sample_precision_train.npy")

feature_names = [
    column
    for column in df.columns
    if column not in ["run_id", "time_step", "time_seconds", "anomaly", "split"]
]
child_parents_pairs = {
    child: sorted(causal_graph.loc[causal_graph["target"] == child, "source"].unique())
    for child in feature_names
}
model_features = {
    child: list(dict.fromkeys(parents + [child]))
    for child, parents in child_parents_pairs.items()
}
process_children = [name for name in feature_names if name in ENDOGENOUS_NAMES]
if set(process_children) != set(ENDOGENOUS_NAMES):
    raise ValueError("Traces are missing endogenous process variables.")

test_run_manifest = runs.loc[
    runs["split"].eq("test") & runs["fault_type"].eq("healthy")
].set_index("run_id")
test_run_ids = test_run_manifest.index.tolist()
df_faultfree = df.loc[df["run_id"].isin(test_run_ids)]
if not test_run_ids:
    raise ValueError("No healthy test runs are available for evaluation.")
if df.groupby("run_id").size().nunique() != 1:
    raise ValueError("All trajectories must have equal length for evaluation.")
n_samples_per_faultfree_ts = int(df.groupby("run_id").size().iloc[0])
n_faultfree_ts = len(test_run_ids)

print(f"Held-out healthy runs: {n_faultfree_ts}")
print(f"Selected lag={max_lag}, window={window}, consecutive alarms={consecutive_alarms}")
print(f"Validation-selected threshold={threshold:.6g}")

Held-out healthy runs: 75
Selected lag=2, window=8, consecutive alarms=5
Validation-selected threshold=2.97599


In [3]:
def create_lagged_data(dataframe, child, lag):
    features = dataframe[model_features[child]]
    lagged = pd.concat(
        [features.shift(step).add_suffix(f"_lag{step}") for step in range(1, lag + 1)],
        axis=1,
    ).iloc[lag:]
    x = np.ones((len(lagged), lagged.shape[1] + 1))
    x[:, 1:] = lagged.to_numpy()
    y = dataframe[child].iloc[lag:].to_numpy()
    return x, y


score_rows = []
for run_id, trajectory in df_faultfree.groupby("run_id", sort=False):
    trajectory = trajectory.sort_values("time_step")
    residuals = np.empty((n_samples_per_faultfree_ts - max_lag, len(process_children)))

    for child_idx, child in enumerate(process_children):
        beta_path = models_dir / f"beta_child_{child}_lag_{max_lag}.npy"
        if not beta_path.is_file():
            raise FileNotFoundError(f"Missing fitted autoregressive model: {beta_path}")
        beta = np.load(beta_path)
        x_test, y_test = create_lagged_data(trajectory, child, max_lag)
        residuals[:, child_idx] = y_test - x_test @ beta

    scores = np.mean((residuals @ sample_precision) * residuals, axis=1)
    smoothed_scores = pd.Series(scores).rolling(window=window, min_periods=window).mean()
    raw_breaches = smoothed_scores.gt(threshold).fillna(False).astype(int)
    alarms = (
        raw_breaches.rolling(
            window=consecutive_alarms,
            min_periods=consecutive_alarms,
        )
        .sum()
        .eq(consecutive_alarms)
        .fillna(False)
        .to_numpy(dtype=int)
    )
    steps = trajectory["time_step"].to_numpy()[max_lag:]
    score_rows.append(
        pd.DataFrame(
            {
                "run_id": run_id,
                "time_step": steps,
                "score": scores,
                "smoothed_score": smoothed_scores.to_numpy(),
                "alarm": alarms,
            }
        )
    )

df_scores = pd.concat(score_rows, ignore_index=True)
df_scores.to_parquet(results_dir / "healthy_test_scores.parquet", index=False)
alarms_by_run = {
    run_id: run_scores["alarm"].to_numpy()
    for run_id, run_scores in df_scores.groupby("run_id", sort=False)
}
print(f"Scored {len(df_scores):,} held-out healthy samples.")

Scored 74,850 held-out healthy samples.


In [4]:
evaluation_mask = np.ones(n_samples_per_faultfree_ts - max_lag, dtype=bool)
evaluation_mask[: window + consecutive_alarms - 2] = False
y_anomaly = np.zeros(n_faultfree_ts * (n_samples_per_faultfree_ts - max_lag), dtype=int)
y_anomaly_pred = np.concatenate(
    [
        alarms_by_run[run_id]
        for run_id in test_run_ids
    ]
)
eval_mask = np.tile(evaluation_mask, n_faultfree_ts)
y_anomaly_eval = y_anomaly[eval_mask]
y_anomaly_pred_eval = y_anomaly_pred[eval_mask]

fp = np.sum((y_anomaly_eval == 0) & (y_anomaly_pred_eval == 1))
tn = np.sum((y_anomaly_eval == 0) & (y_anomaly_pred_eval == 0))
sample_fpr = fp / (fp + tn) if fp + tn else 0.0

y_ts_anomaly = np.zeros(n_faultfree_ts, dtype=int)
y_ts_anomaly_pred = np.array(
    [
        alarms_by_run[run_id].any()
        for run_id in test_run_ids
    ],
    dtype=int,
)
fp_ts = np.sum((y_ts_anomaly == 0) & (y_ts_anomaly_pred == 1))
tn_ts = np.sum((y_ts_anomaly == 0) & (y_ts_anomaly_pred == 0))
trajectory_phase_fpr = fp_ts / (fp_ts + tn_ts) if fp_ts + tn_ts else 0.0

per_run_results = pd.DataFrame(
    [
        {
            "run_id": run_id,
            "false_alarm_samples": int(alarms_by_run[run_id].sum()),
            "trajectory_false_alarm": bool(alarms_by_run[run_id].any()),
        }
        for run_id in test_run_ids
    ]
)
per_run_results["sample_false_alarm_rate"] = (
    per_run_results["false_alarm_samples"]
    / (n_samples_per_faultfree_ts - max_lag - (window + consecutive_alarms - 2))
)
per_run_results.to_csv(results_dir / "healthy_test_false_alarms_by_run.csv", index=False)

df_results = pd.DataFrame(
    [
        {
            "Max Lag": max_lag,
            "Window Size": window,
            "Consecutive Alarms": consecutive_alarms,
            "Threshold": threshold,
            "Sample FPR": sample_fpr,
            "Trajectory-Phase FPR": trajectory_phase_fpr,
            "False Alarm Runs": int(y_ts_anomaly_pred.sum()),
            "Healthy Test Runs": n_faultfree_ts,
            "False Alarm Samples": int(fp),
            "Evaluated Healthy Samples": int(fp + tn),
        }
    ]
)
df_results.to_csv(results_dir / "healthy_test_metrics.csv", index=False)
display(df_results)
display(per_run_results.head())

,Max Lag,Window Size,Consecutive Alarms,Threshold,Sample FPR,Trajectory-Phase FPR,False Alarm Runs,Healthy Test Runs,False Alarm Samples,Evaluated Healthy Samples
0,2,8,5,2.97599,0.0,0.0,0,75,0,74025


,run_id,false_alarm_samples,trajectory_false_alarm,sample_false_alarm_rate
0,run_healthy_00012,0,False,0.0
1,run_healthy_00013,0,False,0.0
2,run_healthy_00020,0,False,0.0
3,run_healthy_00021,0,False,0.0
4,run_healthy_00024,0,False,0.0
